## Sequential Feature Selection
* Performs feature selection by selecting or deselecting features one by one in a greedy manner.
* Uses one of the two approaches
    * 1.Forward selection
        * Starting with a zero feature, it finds one feature that obtains the best cross validation score for an estimator when trained on that feature.
        * Repeats the process by adding a new feature to the set of selected features.
    * 2.Backward selection
        * Starting with all features and removes least important features one by one following the idea of forward selection.
    * Stops when reach the desired number of features. 

* The direction parameter controls whether forward or backward SFS is used.
* In general, forward and backward selection do not yield equivalent results.
* Select the direction that is efficient for the required number of selected features:
* When we want to select 7 out of 10 features, 
    * Forward selection would need to perform 7 iterations.
    * Backward selection would only need to perform 3.
* Backward selection seems to be a reasonable choice here.

* SFS does not require the underlying model to expose a coef_ or feature_importances_ attributes unlike in RFE and SelectFromModel.
* SFS may be slower than RFE and SelectFromModel as it needs to evaluate more models compared to the other two approaches.

    * For example in backward selection, the iteration going from m features to m - 1 features using k-fold cross-validation requires fitting m x k models, while

    * RFE would require only a single fit, and
    * SelectFromModel performs a single fit and requires no iterations.


# Sequential Feature Selection (SFS) Ekdam Detail Mein

## 1. Yeh Kya Hai? (What is SFS?)
Maan lo tum ek cricket team bana rahe ho. Tumhare paas 100 players hain, par tum sabko nahi khila sakte. Tumhe best 11 chahiye jo match jita sakein. Machine Learning me bhi yahi hota hai. Hamare paas bahut saare "features" (columns) hote hain, par saare features important nahi hote. Kuch features model ko confuse kar dete hain (noise), aur jyada features matlab jyada computation time.

**Sequential Feature Selection (SFS)** ek greedy search algorithm hai jo best features ka subset dhoondhta hai. Yeh "greedy" isliye hai kyunki yeh har step par jo sabse best lagta hai use choose kar leta hai, bina yeh soche ki aage chalkar iska kya asar hoga.

Yeh do tareeke se kaam karta hai:
1.  **Forward Selection:** Zero se shuru karo, aur ek-ek karke sabse best feature add karte jao.
2.  **Backward Selection:** Saare features se shuru karo, aur ek-ek karke sabse bekar (least important) feature ko nikalte jao.

---

## 2. Hum Ise Kyu Use Karte Hain? (Why use SFS?)
Tum sochoge ki RFE (Recursive Feature Elimination) ya `SelectFromModel` kyu na use kar lein? Notes me iska exact jawab hai:
*   **Model Agnostic:** RFE aur SelectFromModel ko aise models chahiye hote hain jo features ki importance bata sakein (jaise `coef_` Linear Regression me, ya `feature_importances_` Random Forest me). Par kya hoga agar tum KNN (K-Nearest Neighbors) use kar rahe ho jisme aise koi attributes nahi hote? Wahan SFS hero banta hai kyunki ise underlying model ke internal coefficients ki jarurat nahi padti. Yeh bas direct cross-validation score check karta hai.

---

## 3. Maths aur Logic Iske Piche (How it works & The Math)
SFS ka sabse bada drawback iski speed hai. Chalo samajhte hain kyu:

**Iteration aur Efficiency ka Khel:**
Maan lo total features $N = 10$ hain, aur tumhe $k = 7$ features select karne hain.
*   **Forward SFS:** Ise $7$ features add karne hain, toh yeh $7$ iterations (steps) lega.
*   **Backward SFS:** Ise $10$ se shuru karke $7$ tak aana hai, matlab sirf $3$ features hatane hain. Yeh sirf $3$ iterations lega.
*   **Rule of Thumb:** Agar $k < N/2$, toh Forward fast hoga. Agar $k > N/2$, toh Backward fast hoga.

**Kitne models train hote hain? (The Cost):**
Backward selection me $m$ se $m-1$ features jane ke liye, $k$-fold cross-validation ke sath $m \times k$ models train hote hain. Aisa kyu?
*   Maan lo current step me tumhare paas $m = 5$ features hain. Tumhe 1 feature hatana hai.
*   Algorithm pehle feature 1 ko hatayega, baaki 4 bache. Un 4 par $k$-fold CV lagayega (matlab $k$ models train honge).
*   Phir feature 2 ko hatayega, baaki 4 par $k$-fold CV lagayega ($k$ models train).
*   Yeh har $m$ feature ke liye karega. Total fits = $m \times k$.
*   Isi wajah se SFS thoda slow hota hai jab datasets bahut bade hote hain. RFE me sirf ek single fit me pata chal jata hai ki kaunsa feature hatana hai.

---



In [1]:
## 4. Implementation Code
# Scikit-Learn ke sath implementation. Yahan KNN use kiya hai kyunki KNN RFE support nahi karta, toh SFS ka real use-case dikhega.


# Import necessary libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score
import time

# 1. Dataset Load karna
data = load_breast_cancer()
X, y = data.data, data.target # Total 30 features hain isme

# Data split for training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Original shape of dataset: {X_train.shape}")

# 2. Base Model Define karna
# KNN me koi 'coef_' nahi hota, isliye RFE yahan kaam nahi karega, SFS kaam karega.
knn = KNeighborsClassifier(n_neighbors=3)

# 3. Forward SFS Setup
print("\n--- Starting Forward SFS ---")
start_time = time.time()
sfs_forward = SequentialFeatureSelector(
    estimator=knn, 
    n_features_to_select=10, 
    direction='forward',
    cv=3,           # 3-fold cross validation
    n_jobs=-1       # Saare CPU cores use karo fast processing ke liye
)

sfs_forward.fit(X_train, y_train)
forward_time = time.time() - start_time
X_train_forward = sfs_forward.transform(X_train)
X_test_forward = sfs_forward.transform(X_test)

print(f"Forward SFS Time: {forward_time:.2f} seconds")
print(f"New shape after Forward SFS: {X_train_forward.shape}")

# 4. Backward SFS Setup
print("\n--- Starting Backward SFS ---")
start_time = time.time()
sfs_backward = SequentialFeatureSelector(
    estimator=knn, 
    n_features_to_select=10, 
    direction='backward',
    cv=3,
    n_jobs=-1
)

sfs_backward.fit(X_train, y_train)
backward_time = time.time() - start_time
X_train_backward = sfs_backward.transform(X_train)

print(f"Backward SFS Time: {backward_time:.2f} seconds")
print(f"New shape after Backward SFS: {X_train_backward.shape}")

# 5. Model Evaluation
knn.fit(X_train_forward, y_train)
y_pred = knn.predict(X_test_forward)
print(f"\nAccuracy with Forward SFS features: {accuracy_score(y_test, y_pred)*100:.2f}%")

Original shape of dataset: (455, 30)

--- Starting Forward SFS ---
Forward SFS Time: 18.78 seconds
New shape after Forward SFS: (455, 10)

--- Starting Backward SFS ---
Backward SFS Time: 5.42 seconds
New shape after Backward SFS: (455, 10)

Accuracy with Forward SFS features: 92.11%


## 5. Code Line-by-Line Explanation:
*   **`load_breast_cancer`**: Basic dataset liya jisme 30 features hain taaki processing fast ho aur difference dikhe.
*   **`KNeighborsClassifier(n_neighbors=3)`**: Base model. KNN use kiya kyunki yeh SFS ki power dikhata hai (RFE support nahi karta).
*   **`SequentialFeatureSelector(...)`**: Asali SFS object.
    *   `estimator=knn`: Kis model ki accuracy check karni hai.
    *   `n_features_to_select=10`: 30 me se best 10 features chahiye.
    *   `direction='forward'` / `'backward'`: Add karna hai ya remove.
    *   `cv=3`: Cross-validation folds (Model overfit na ho isliye jaruri hai).
    *   `n_jobs=-1`: SFS compute-heavy hai, `-1` se saare CPU cores parallel kaam karenge.
*   **`.fit(X_train, y_train)`**: Best features dhoondhne ka loop yahan chalta hai.
*   **`.transform(X_train)`**: Bekar columns drop karke naya data deta hai jisme sirf best 10 columns hote hain.